# CodonFM SAE — Automated Feature Interpretation

The previous notebooks give us *quantitative* descriptions of features: which codons they promote, what GSEA terms they're enriched for, their activation frequency. But these are still fragmented signals — a human has to synthesize "this feature promotes GCC/GCT, fires on ribosomal protein genes, and has high tAI correlation" into "this feature detects optimally-translated codons in highly-expressed housekeeping genes."

**Auto-interpretation** automates that synthesis step by sending all available evidence about a feature to a large language model and asking it to produce:
1. A **description** (2-3 sentences) of what the feature detects and why.
2. A **label** (one concise phrase) suitable for a dashboard tooltip.
3. A **confidence score** (0.0-1.0) reflecting how interpretable the pattern is.

The prompt includes:
- **Top promoted/suppressed codons** from the decoder weight projection through the LM head
- **Top activating sequences** with per-codon highlighting (codons where the feature fires strongly are marked)
- **Gene metadata** (gene names, pathogenicity labels, variant info) for each example
- **GSEA enrichment context** (if available from notebook 03) — the top enriched biological terms

This is expensive (one LLM call per feature), so we demo on a small subset here. The `analyze.py` script handles production-scale runs with checkpointing and parallelism.

## Setup

In [ ]:
# Config
SAE_CHECKPOINT = "../outputs/1b_layer16/checkpoints/checkpoint_final.pt"
MODEL_PATH = "../../../../../../../checkpoints/NV-CodonFM-Encodon-TE-Cdwt-1B-v1"
CSV_PATH = "../../../../../../../codonfm/data/codonfm_ref_only.csv"
LAYER = 16
CONTEXT_LENGTH = 2048
BATCH_SIZE = 8
DEVICE = "cuda"

# Auto-interp specific
LLM_PROVIDER = "nvidia-internal"  # Options: "anthropic", "openai", "nim", "nvidia-internal"
NUM_FEATURES_TO_INTERPRET = 20  # Small subset for demo
NUM_SEQUENCES = 500  # Enough to find good examples

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import torch


_REPO_ROOT = Path("..").resolve().parent.parent.parent.parent.parent
_CODONFM_TE_DIR = _REPO_ROOT / "recipes" / "codonfm_ptl_te"
sys.path.insert(0, str(_CODONFM_TE_DIR))
sys.path.insert(0, str(Path("..").resolve()))

from codonfm_sae.data import read_codon_csv
from sae.architectures import TopKSAE
from sae.utils import set_seed
from src.data.preprocess.codon_sequence import process_item
from src.inference.encodon import EncodonInference


set_seed(42)
device = DEVICE if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Load SAE, Model, and Data

In [ ]:
ckpt = torch.load(SAE_CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = ckpt["model_state_dict"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.removeprefix("module."): v for k, v in state_dict.items()}

input_dim = ckpt.get("input_dim") or state_dict["encoder.weight"].shape[1]
hidden_dim = ckpt.get("hidden_dim") or state_dict["encoder.weight"].shape[0]
model_config = ckpt.get("model_config", {})
top_k = model_config.get("top_k")

sae = TopKSAE(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    top_k=top_k,
    normalize_input=model_config.get("normalize_input", False),
)
sae.load_state_dict(state_dict)
sae = sae.eval().to(device)

print(f"SAE: {input_dim} -> {hidden_dim:,} features (top-{top_k})")

In [ ]:
inference = EncodonInference(
    model_path=MODEL_PATH,
    task_type="embedding_prediction",
    use_transformer_engine=True,
)
inference.configure_model()
inference.model.to(device).eval()

num_layers = len(inference.model.model.layers)
target_layer = LAYER if LAYER >= 0 else num_layers + LAYER
print(f"Encodon: {num_layers} layers, target layer {target_layer}")

In [ ]:
records = read_codon_csv(CSV_PATH, max_sequences=NUM_SEQUENCES, max_codons=CONTEXT_LENGTH - 2)
sequences = [r.sequence for r in records]
print(f"Loaded {len(sequences)} sequences")

## Compute Vocabulary Logits

Before building prompts, we need to know which codons each feature promotes and suppresses. This comes from projecting the SAE decoder weights through the model's LM head — the same vocabulary projection the model uses to predict the next codon.

In [ ]:
CODON_TO_AA = {
    "TTT": "F",
    "TTC": "F",
    "TTA": "L",
    "TTG": "L",
    "CTT": "L",
    "CTC": "L",
    "CTA": "L",
    "CTG": "L",
    "ATT": "I",
    "ATC": "I",
    "ATA": "I",
    "ATG": "M",
    "GTT": "V",
    "GTC": "V",
    "GTA": "V",
    "GTG": "V",
    "TCT": "S",
    "TCC": "S",
    "TCA": "S",
    "TCG": "S",
    "CCT": "P",
    "CCC": "P",
    "CCA": "P",
    "CCG": "P",
    "ACT": "T",
    "ACC": "T",
    "ACA": "T",
    "ACG": "T",
    "GCT": "A",
    "GCC": "A",
    "GCA": "A",
    "GCG": "A",
    "TAT": "Y",
    "TAC": "Y",
    "TAA": "*",
    "TAG": "*",
    "CAT": "H",
    "CAC": "H",
    "CAA": "Q",
    "CAG": "Q",
    "AAT": "N",
    "AAC": "N",
    "AAA": "K",
    "AAG": "K",
    "GAT": "D",
    "GAC": "D",
    "GAA": "E",
    "GAG": "E",
    "TGT": "C",
    "TGC": "C",
    "TGA": "*",
    "TGG": "W",
    "CGT": "R",
    "CGC": "R",
    "CGA": "R",
    "CGG": "R",
    "AGT": "S",
    "AGC": "S",
    "AGA": "R",
    "AGG": "R",
    "GGT": "G",
    "GGC": "G",
    "GGA": "G",
    "GGG": "G",
}

tokenizer = inference.tokenizer
codon_tokens = {}
for codon in CODON_TO_AA:
    tok_id = tokenizer.token_to_id(codon)
    if tok_id is not None:
        codon_tokens[codon] = tok_id

# Project decoder through LM head
encodon = inference.model.model
lm_head = encodon.cls
W_dec = sae.decoder.weight.to(device)

with torch.no_grad():
    logits = lm_head(W_dec.T)  # (n_features, vocab_size)

# Baseline logits (from mean activation)
mean_acts = sae.pre_bias.data.float().to(device) if hasattr(sae, "pre_bias") else torch.zeros(input_dim, device=device)
with torch.no_grad():
    baseline = lm_head(mean_acts.unsqueeze(0)).squeeze(0)

logit_diff = logits - baseline.unsqueeze(0)  # (n_features, vocab_size)

print(f"Computed vocab logit diffs: {logit_diff.shape}")

## Prepare Feature Examples

For each feature we want to interpret, we need to find the sequences where it fires most strongly and format them for the prompt. Each example shows the codon sequence with high-activation positions highlighted using `***markers***`.

In [ ]:
from tqdm import tqdm


# First, find the most frequently-firing features to interpret
n_features = sae.hidden_dim
fire_counts = np.zeros(n_features, dtype=np.int64)
max_activations = np.zeros(n_features, dtype=np.float32)
# Also store per-sequence max acts for finding top examples
seq_max_acts = np.zeros((len(sequences), n_features), dtype=np.float32)

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc="Computing activations"):
        batch_seqs = sequences[i : i + BATCH_SIZE]
        items = [process_item(s, context_length=CONTEXT_LENGTH, tokenizer=tokenizer) for s in batch_seqs]
        batch = {
            "input_ids": torch.tensor(np.stack([it["input_ids"] for it in items])).to(device),
            "attention_mask": torch.tensor(np.stack([it["attention_mask"] for it in items])).to(device),
        }
        out = inference.model(batch, return_hidden_states=True)
        hidden = out.all_hidden_states[LAYER]

        for j, it in enumerate(items):
            seq_len = it["attention_mask"].sum()
            emb = hidden[j, 1 : seq_len - 1, :].float()
            codes = sae.encode(emb)
            active = (codes > 0).cpu().numpy()
            fire_counts += active.sum(axis=0)
            seq_max = codes.max(dim=0).values.cpu().numpy()
            np.maximum(max_activations, seq_max, out=max_activations)
            seq_max_acts[i + j] = seq_max

        del out, hidden, batch
        torch.cuda.empty_cache()

# Select top features by activation frequency
alive = fire_counts > 0
alive_indices = np.where(alive)[0]
sorted_by_freq = alive_indices[np.argsort(fire_counts[alive_indices])[::-1]]
features_to_interpret = sorted_by_freq[:NUM_FEATURES_TO_INTERPRET].tolist()

print(f"{alive.sum()} alive features, interpreting top {len(features_to_interpret)} by frequency")

In [ ]:
# For each feature, get the top-5 sequences and extract per-codon activations
N_EXAMPLES = 5

feature_examples = {}  # feat_idx -> list of (max_act, seq_idx, per_codon_acts)

for feat_idx in tqdm(features_to_interpret, desc="Collecting examples"):
    # Find top sequences for this feature
    top_seq_indices = np.argsort(seq_max_acts[:, feat_idx])[::-1][:N_EXAMPLES]

    examples = []
    for seq_idx in top_seq_indices:
        if seq_max_acts[seq_idx, feat_idx] == 0:
            continue
        seq = sequences[seq_idx]
        items = [process_item(seq, context_length=CONTEXT_LENGTH, tokenizer=tokenizer)]
        batch = {
            "input_ids": torch.tensor(np.stack([it["input_ids"] for it in items])).to(device),
            "attention_mask": torch.tensor(np.stack([it["attention_mask"] for it in items])).to(device),
        }
        with torch.no_grad():
            out = inference.model(batch, return_hidden_states=True)
            hidden = out.all_hidden_states[LAYER]
            seq_len = items[0]["attention_mask"].sum()
            emb = hidden[0, 1 : seq_len - 1, :].float()
            _, codes = sae(emb)
            acts = codes[:, feat_idx].cpu().numpy()

        examples.append((float(acts.max()), int(seq_idx), acts))
        del out, batch

    feature_examples[feat_idx] = examples

torch.cuda.empty_cache()
print(f"Collected examples for {len(feature_examples)} features")

## Build the Interpretation Prompt

This is the core of auto-interp: assembling all available evidence into a structured prompt. The LLM receives:

1. **Context** about the model (CodonFM, a DNA codon language model)
2. **Decoder logit analysis** — which codons the feature promotes/suppresses
3. **Top activating sequences** with highlighted codons and metadata
4. **GSEA enrichment** (if available) — biological annotations
5. **Output format** instructions

Let's build a prompt for one feature to see what it looks like.

In [ ]:
def build_prompt(
    feat_idx, examples, logit_diff_tensor, codon_tokens_map, records_list, sequences_list, gsea_context=None
):
    """Build the auto-interp prompt for a single feature."""
    # Top promoted/suppressed codons
    feat_logits = {c: float(logit_diff_tensor[feat_idx, tid]) for c, tid in codon_tokens_map.items()}
    sorted_codons = sorted(feat_logits.items(), key=lambda x: x[1], reverse=True)
    top_pos = sorted_codons[:8]
    top_neg = sorted_codons[-8:][::-1]

    pos_str = ", ".join(f"{c}({CODON_TO_AA[c]})={v:.2f}" for c, v in top_pos)
    neg_str = ", ".join(f"{c}({CODON_TO_AA[c]})={v:.2f}" for c, v in top_neg)

    # Format example sequences
    examples_parts = []
    for rank, (max_act, seq_idx, acts) in enumerate(examples[:5]):
        seq = sequences_list[seq_idx]
        vl = len(acts)
        codons = [seq[j * 3 : (j + 1) * 3] for j in range(vl)]

        threshold = np.percentile(acts[acts > 0], 80) if (acts > 0).sum() > 0 else 0
        marked = []
        for j, (codon, act) in enumerate(zip(codons, acts)):
            aa = CODON_TO_AA.get(codon.upper(), "?")
            if act > threshold:
                marked.append(f"***{codon}({aa})***")
            else:
                marked.append(f"{codon}({aa})")

        # Metadata
        meta_str = ""
        if records_list and seq_idx < len(records_list):
            m = records_list[seq_idx].metadata
            meta_parts = []
            gene = m.get("gene")
            if gene:
                meta_parts.append(f"gene={gene}")
            is_path = m.get("is_pathogenic")
            if is_path is not None:
                meta_parts.append(f"pathogenic={is_path}")
            if meta_parts:
                meta_str = f"  [{', '.join(meta_parts)}]"

        # Show first 60 codons to keep prompt manageable
        codon_str = " ".join(marked[:60])
        if len(marked) > 60:
            codon_str += f" ... ({len(marked)} codons total)"

        examples_parts.append(f"Example {rank + 1} (max_act={max_act:.2f}){meta_str}:\n{codon_str}")

    examples_str = "\n\n".join(examples_parts)

    # GSEA context
    gsea_str = "Not available (run notebook 03 first)"
    if gsea_context and str(feat_idx) in gsea_context:
        ctx = gsea_context[str(feat_idx)]
        gsea_str = ctx if isinstance(ctx, str) else json.dumps(ctx, indent=2)

    prompt = f"""Analyze this sparse autoencoder feature from a DNA codon language model (CodonFM).

CodonFM is trained on coding DNA sequences (codons = 3-nucleotide triplets that encode amino acids).
This SAE feature was learned from the model's internal representations. Your task is to identify
what biological pattern this feature detects.

Top promoted codons (decoder logits, higher = feature promotes this codon):
{pos_str}

Top suppressed codons (decoder logits, lower = feature suppresses this codon):
{neg_str}

Top activating sequences (***highlighted*** = high activation codons):

{examples_str}

Gene-level GSEA enrichment:
{gsea_str}

Based on all available evidence, describe what this feature detects.
Consider: codon usage bias, amino acid preferences, gene family specificity,
sequence composition patterns, biological pathway associations.

Format your response EXACTLY as:
Description: <2-3 sentences explaining the activation pattern and its biological significance>
Label: <one concise phrase, max 8 words>
Confidence: <0.00 to 1.00, how confident you are in this interpretation>"""

    return prompt


# Build and display a prompt for the first feature
example_feat = features_to_interpret[0]
example_prompt = build_prompt(
    example_feat,
    feature_examples[example_feat],
    logit_diff,
    codon_tokens,
    records,
    sequences,
)
print(f"Prompt for feature {example_feat} ({len(example_prompt)} chars):")
print("=" * 80)
print(example_prompt)
print("=" * 80)

## Run LLM Interpretation

Send the prompt to an LLM. The `sae.autointerp` module provides clients for multiple providers:
- `AnthropicClient` — Claude (requires `ANTHROPIC_API_KEY`)
- `OpenAIClient` — GPT-4 (requires `OPENAI_API_KEY`)
- `NIMClient` — NVIDIA NIM (requires `NIM_API_KEY`)
- `NVIDIAInternalClient` — Internal NVIDIA endpoint (requires `CLAUDE_SONNET_INFERENCE_API_KEY`)

Uncomment the appropriate client below and ensure the environment variable is set.

In [ ]:
# Uncomment the client for your LLM provider:
# client = NVIDIAInternalClient()       # Requires CLAUDE_SONNET_INFERENCE_API_KEY env var
# client = AnthropicClient()             # Requires ANTHROPIC_API_KEY env var
# client = OpenAIClient()                # Requires OPENAI_API_KEY env var
# client = NIMClient()                   # Requires NIM_API_KEY env var

# For demo purposes, we'll show the prompt without calling the LLM.
# To actually run interpretation, uncomment one of the clients above and
# uncomment the code in the next cell.
print("LLM client setup. Uncomment the appropriate client above to run interpretation.")

In [ ]:
# Load GSEA context if available (from notebook 03)
gsea_context = None
gsea_report_path = Path("../outputs/1b_layer16/gene_enrichment/gene_enrichment_report.json")
if gsea_report_path.exists():
    with open(gsea_report_path) as f:
        gsea_data = json.load(f)
    gsea_context = {}
    for entry in gsea_data.get("per_feature", []):
        idx = str(entry["feature_idx"])
        ob = entry.get("overall_best")
        if ob:
            gsea_context[idx] = f"{ob['term_name']} ({ob['database']}, FDR={ob['fdr']:.4f})"
    print(f"Loaded GSEA context for {len(gsea_context)} features")
else:
    print("No GSEA report found. Run notebook 03 first for richer prompts.")

In [ ]:
# Uncomment to run LLM interpretation on a few features:
#
# results = {}
# for feat_idx in features_to_interpret[:5]:  # Start with just 5
#     prompt = build_prompt(
#         feat_idx,
#         feature_examples[feat_idx],
#         logit_diff,
#         codon_tokens,
#         records,
#         sequences,
#         gsea_context=gsea_context,
#     )
#     response = client.generate(prompt)
#     results[feat_idx] = response.text
#     print(f"\nFeature {feat_idx}:")
#     print(response.text)
#     print("-" * 40)

print("Uncomment the cell above to run LLM interpretation.")

## Parse Results

The LLM response follows a structured format. We extract the label, description, and confidence score from each response.

In [ ]:
import re


def parse_interp_response(text):
    """Extract description, label, and confidence from LLM response."""
    description = ""
    label = ""
    confidence = 0.0

    for line in text.strip().split("\n"):
        line = line.strip()
        if line.lower().startswith("description:"):
            description = line.split(":", 1)[1].strip()
        elif line.lower().startswith("label:"):
            label = line.split(":", 1)[1].strip()
        elif line.lower().startswith("confidence:"):
            try:
                confidence = float(re.search(r"[\d.]+", line.split(":", 1)[1]).group())
            except (ValueError, AttributeError):
                confidence = 0.0

    return {"description": description, "label": label, "confidence": confidence}


# Example parsing (using a mock response since we may not have run the LLM)
mock_response = """Description: This feature detects GC-rich optimal codons (GCC, GCG, CTG) that are preferentially used in highly expressed human genes. The highlighted positions cluster in regions encoding alanine and leucine with strong codon usage bias toward tRNA-abundant codons.
Label: GC-rich optimal codon usage
Confidence: 0.85"""

parsed = parse_interp_response(mock_response)
print(f"Label:       {parsed['label']}")
print(f"Confidence:  {parsed['confidence']}")
print(f"Description: {parsed['description']}")

## Example Results

Below is what interpreted features look like. Each feature gets a human-readable label that can be displayed in the dashboard, along with a confidence score indicating how interpretable the pattern is.

Features with high confidence (>0.7) typically have clear codon usage or gene family patterns. Low confidence features may detect more subtle or combinatorial patterns that are harder to describe in words.

In [ ]:
# If you ran the LLM interpretation, parse and display results:
#
# parsed_results = {}
# for feat_idx, text in results.items():
#     parsed = parse_interp_response(text)
#     parsed_results[feat_idx] = parsed
#     print(f"Feature {feat_idx}:")
#     print(f"  Label:       {parsed['label']}")
#     print(f"  Confidence:  {parsed['confidence']}")
#     print(f"  Description: {parsed['description'][:100]}...")
#     print()
#
# # Save results
# output_dir = Path("../outputs/1b_layer16/analysis")
# output_dir.mkdir(parents=True, exist_ok=True)
# with open(output_dir / "auto_interp_results.json", "w") as f:
#     json.dump({str(k): v for k, v in parsed_results.items()}, f, indent=2)
# print(f"Saved {len(parsed_results)} interpretations")

print("Uncomment the cells above after running LLM interpretation.")

## Scaling Up

Interpreting 20 features is fine for exploration, but a full SAE may have thousands of alive features. The `analyze.py` script handles this at scale with:
- **Checkpointing**: saves progress after each batch so you can resume if interrupted
- **Parallel LLM calls**: uses `--auto-interp-workers` to send multiple requests concurrently
- **GSEA context injection**: pass `--gsea-report` to include enrichment data in prompts
- **Dashboard integration**: pass `--dashboard-dir` to write labels directly to the atlas parquet

In [ ]:
print("""To run auto-interp on all features:

  python scripts/analyze.py \\
      --checkpoint outputs/1b_layer16/checkpoints/checkpoint_final.pt \\
      --model-path $MODEL_PATH \\
      --csv-path $CSV_PATH \\
      --layer 16 --auto-interp \\
      --llm-provider nvidia-internal \\
      --gsea-report outputs/1b_layer16/gene_enrichment/gene_enrichment_report.json \\
      --auto-interp-workers 8 \\
      --output-dir outputs/1b_layer16/analysis
""")

## Next Steps

With all analysis notebooks complete, you have:

1. **01_quickstart** — SAE health checks, loss recovered, basic feature stats
2. **02_codon_analysis** — Codon usage metrics (CAI, tAI, RSCU) per feature
3. **03_gene_enrichment** — GSEA labels, gene families, pLI scores
4. **04_dashboard** — Interactive visualization export
5. **05_auto_interp** — LLM-generated natural language descriptions

Together these provide a comprehensive picture of what each SAE feature has learned about codon biology.